In [23]:
import pandas as pd
import numpy as np
from scipy.linalg import solve


In [24]:
b = pd.read_csv("nl20_buses.csv")
l = pd.read_csv("nl20_lines.csv")
print(b)

              name  v_nom  type         x          y  carrier  unit location  \
0            NL0 0  380.0   NaN  4.580489  51.975314       AC   NaN      NaN   
1            NL0 1  380.0   NaN  6.925060  53.288277       AC   NaN      NaN   
2           NL0 10  380.0   NaN  6.759179  52.248853       AC   NaN      NaN   
3           NL0 11  380.0   NaN  6.499371  53.111367       AC   NaN      NaN   
4           NL0 12  380.0   NaN  5.663999  51.926845       AC   NaN      NaN   
5           NL0 13  380.0   NaN  4.265989  51.839411       AC   NaN      NaN   
6           NL0 14  380.0   NaN  6.251938  51.982219       AC   NaN      NaN   
7           NL0 15  380.0   NaN  4.987569  52.160521       AC   NaN      NaN   
8           NL0 16  380.0   NaN  5.925228  53.124994       AC   NaN      NaN   
9           NL0 17  380.0   NaN  4.265558  51.999116       AC   NaN      NaN   
10          NL0 18  380.0   NaN  4.687186  52.423286       AC   NaN      NaN   
11          NL0 19  380.0   NaN  4.04413

In [25]:
# clean the data, keeping only columns we need and get rid of empty rows
buses = b[["name"]].copy()
# drop H2 and battery buses, not relevant for our analysis
buses = buses[~buses["name"].str.contains("H2|battery", case=False, na=False)]


# data needed from lines: name, bus0, bus1, reactance x and capacity s_nom
lines = l[["name", "bus0", "bus1", "x", "r", "s_nom"]].copy()
# drop lines if value is missing or 0 for any of the columns
lines = lines.dropna(subset=["name", "bus0", "bus1", "x", "r", "s_nom"])
lines = lines[(lines["x"] != 0) & (lines["r"] != 0) & (lines["s_nom"] != 0)]
print(lines)

    name    bus0    bus1          x         r        s_nom
0      0   NL0 0  NL0 15   4.257970  0.519264  3396.205223
1      1   NL0 0  NL0 17   2.672492  0.325914  3396.205223
2     10  NL0 12  NL0 14   5.013215  0.611368  3396.205223
3     11  NL0 12   NL0 2   4.471080  0.545254  3396.205223
4     12  NL0 13  NL0 19   4.380563  0.534215  3396.205223
5     13  NL0 13   NL0 8   4.578888  0.558401  3396.205223
6     14  NL0 15   NL0 4   3.061643  0.373371  3396.205223
7     15  NL0 16   NL0 7   9.197854  1.833459   983.112038
8     16  NL0 17  NL0 19   2.574182  0.313925  3396.205223
9     17  NL0 18   NL0 4   2.224974  0.271338  3396.205223
10    18   NL0 2   NL0 6   6.703120  0.817453  3396.205223
11    19   NL0 3   NL0 7   2.409672  0.356019  4379.317262
12     2   NL0 0  NL0 18   6.191831  0.755102  3396.205223
13    20   NL0 4   NL0 7   6.734747  0.821311  3396.205223
15    22   NL0 5   NL0 8   6.664696  0.812768  3396.205223
16    23   NL0 6   NL0 9   5.247236  0.639907  3396.2052

In [26]:
# reactance-resistance ratio of the lines to check suitability for DC power flow approximation (neglectable ratio > 4)
avg_xr_ratio = (lines["x"] / lines["r"]).mean()
min_xr_ratio = (lines["x"] / lines["r"]).min()
max_xr_ratio = (lines["x"] / lines["r"]).max()
print(f"Average x/r ratio: {avg_xr_ratio:.2f}")
print(f"Minimum x/r ratio: {min_xr_ratio:.2f}")     
print(f"Maximum x/r ratio: {max_xr_ratio:.2f}")

Average x/r ratio: 7.74
Minimum x/r ratio: 5.02
Maximum x/r ratio: 8.20


In [27]:
# indices for buses and lines
buses["bus_idx"] = np.arange(len(buses))
lines["line_idx"] = np.arange(len(lines))
# create a mapping from bus name to bus index
bus_name_to_idx = dict(zip(buses["name"], buses["bus_idx"]))

# map line endpoints to bus indices
lines["bus0_idx"] = lines["bus0"].map(bus_name_to_idx)
lines["bus1_idx"] = lines["bus1"].map(bus_name_to_idx)
print(lines[["name", "bus0", "bus1", "bus0_idx", "bus1_idx"]])

    name    bus0    bus1  bus0_idx  bus1_idx
0      0   NL0 0  NL0 15         0         7
1      1   NL0 0  NL0 17         0         9
2     10  NL0 12  NL0 14         4         6
3     11  NL0 12   NL0 2         4        12
4     12  NL0 13  NL0 19         5        11
5     13  NL0 13   NL0 8         5        18
6     14  NL0 15   NL0 4         7        14
7     15  NL0 16   NL0 7         8        17
8     16  NL0 17  NL0 19         9        11
9     17  NL0 18   NL0 4        10        14
10    18   NL0 2   NL0 6        12        16
11    19   NL0 3   NL0 7        13        17
12     2   NL0 0  NL0 18         0        10
13    20   NL0 4   NL0 7        14        17
15    22   NL0 5   NL0 8        15        18
16    23   NL0 6   NL0 9        16        19
17    24   NL0 8   NL0 9        18        19
18     3   NL0 0   NL0 8         0        18
19     4   NL0 1  NL0 11         1         3
20     5   NL0 1   NL0 3         1        13
21     6  NL0 10  NL0 14         2         6
22     7  

In [28]:
# incidence matrix A, where A[l, bus0] = +1 (from) and A[l, bus1] = -1 (to)
num_buses = len(buses)
num_lines = len(lines)
A = np.zeros((num_lines, num_buses), dtype=int)

for _, row in lines.iterrows():
    bus0 = row["bus0_idx"]
    bus1 = row["bus1_idx"]
    l = row["line_idx"]
    A[l, bus0] = 1
    A[l, bus1] = -1
print("Incidence matrix shape:", A.shape)
print(buses[["bus_idx", "name"]].head(10))
display(pd.DataFrame(A[:24, :20]))
row_sums = A.sum(axis=1)
print("\nUnique row sums of A (should be 0):", np.unique(row_sums))

Incidence matrix shape: (24, 20)
   bus_idx    name
0        0   NL0 0
1        1   NL0 1
2        2  NL0 10
3        3  NL0 11
4        4  NL0 12
5        5  NL0 13
6        6  NL0 14
7        7  NL0 15
8        8  NL0 16
9        9  NL0 17


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,-1,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0
6,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,0
7,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,-1,0,0
8,0,0,0,0,0,0,0,0,0,1,0,-1,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,1,0,0,0,-1,0,0,0,0,0



Unique row sums of A (should be 0): [0]


In [16]:
# save the ordering tables for later export
buses[["bus_idx", "name"]].to_csv("bus_order.csv", index=False)
lines[["line_idx", "name", "bus0", "bus1", "x", "s_nom"]].to_csv("line_order.csv", index=False)

In [17]:
# define reference bus
ref_bus_name = buses.loc[0, "name"]  # choose the first bus as reference
ref_bus_idx = int(buses.loc[buses["name"] == ref_bus_name, "bus_idx"].iloc[0])  # explanation: get the index of the reference bus
print(f"Reference bus: {ref_bus_name} index({ref_bus_idx})")

Reference bus: NL0 0 index(0)


In [29]:
# line susceptance vector b = 1/x
x = lines["x"].to_numpy(dtype=float)
b = 1 / x

# build diagonal matrix B = diag(b)
B = np.diag(b)

# remove the reference bus column from A to get A_red
A_red = np.delete(A, ref_bus_idx, axis=1)

# compute reduced bus susceptance matrix
B_red = A_red.T @ B @ A_red

print("Reduced bus susceptance matrix B_red shape:", B_red.shape)
print("A_red shape:", A_red.shape)

Reduced bus susceptance matrix B_red shape: (19, 19)
A_red shape: (24, 19)


In [30]:
# solve (instead of invert)
PTDF_red = np.linalg.solve(B_red, (B @ A_red).T).T
print("PTDF_red shape:", PTDF_red.shape)

PTDF_red shape: (24, 19)


In [31]:
# insert zero column back at the slack position
PTDF = np.insert(PTDF_red, ref_bus_idx, 0.0, axis=1)
print("Final PTDF shape:", PTDF.shape)


ptdf_df = pd.DataFrame(
    PTDF,
    index=lines["name"],
    columns=buses["name"]
)

# change column names to Bus, row names to Line for easier reference
ptdf_df.columns.name = "Bus"
ptdf_df.index.name = "Line"

display(ptdf_df.iloc[:20, :20])

ptdf_df.to_csv("ptdf_matrix.csv")

Final PTDF shape: (24, 20)


Bus,NL0 0,NL0 1,NL0 10,NL0 11,NL0 12,NL0 13,NL0 14,NL0 15,NL0 16,NL0 17,NL0 18,NL0 19,NL0 2,NL0 3,NL0 4,NL0 5,NL0 6,NL0 7,NL0 8,NL0 9
Line,,,,,,,,,,,,,,,,,,,,
0,0.0,-0.409339,-0.342296,-0.411037,-0.234273,-0.010669,-0.285261,-0.706256,-0.417016,-0.002962,-0.364179,-0.005814,-0.188799,-0.404535,-0.495044,-0.015743,-0.120624,-0.426547,-0.015743,-0.067256
1,0.0,-0.025570,-0.039228,-0.025224,-0.061233,-0.393983,-0.050846,-0.004719,-0.024006,-0.831771,-0.005967,-0.669731,-0.070496,-0.026549,-0.008111,-0.105750,-0.084384,-0.022065,-0.105750,-0.095256
10,0.0,-0.234681,-0.360029,-0.231508,0.438007,0.019946,-0.466664,-0.043306,-0.220328,0.005537,-0.054766,0.010870,0.352987,-0.243663,-0.074445,0.029433,0.225523,-0.202510,0.029433,0.125745
11,0.0,0.234681,0.360029,0.231508,0.561993,-0.019946,0.466664,0.043306,0.220328,-0.005537,0.054766,-0.010870,-0.352987,0.243663,0.074445,-0.029433,-0.225523,0.202510,-0.029433,-0.125745
12,0.0,0.025570,0.039228,0.025224,0.061233,0.393983,0.050846,0.004719,0.024006,-0.168229,0.005967,-0.330269,0.070496,0.026549,0.008111,0.105750,0.084384,0.022065,0.105750,0.095256
13,0.0,-0.025570,-0.039228,-0.025224,-0.061233,0.606017,-0.050846,-0.004719,-0.024006,0.168229,-0.005967,0.330269,-0.070496,-0.026549,-0.008111,-0.105750,-0.084384,-0.022065,-0.105750,-0.095256
14,0.0,-0.409339,-0.342296,-0.411037,-0.234273,-0.010669,-0.285261,0.293744,-0.417016,-0.002962,-0.364179,-0.005814,-0.188799,-0.404535,-0.495044,-0.015743,-0.120624,-0.426547,-0.015743,-0.067256
15,0.0,0.273984,0.065197,0.343567,0.044622,0.002032,0.054334,-0.004412,0.588690,0.000564,-0.005579,0.001107,0.035961,0.077052,-0.007584,0.002999,0.022975,-0.020631,0.002999,0.012810
16,0.0,-0.025570,-0.039228,-0.025224,-0.061233,-0.393983,-0.050846,-0.004719,-0.024006,0.168229,-0.005967,-0.669731,-0.070496,-0.026549,-0.008111,-0.105750,-0.084384,-0.022065,-0.105750,-0.095256


In [32]:
# sanity check: apply a test transfer and see if the resulting flows make sense
# inject +1 at one bus and withdraw -1 at another bus
inj_bus = buses.loc[10, "name"] if len(buses) > 1 else buses.loc[1, "name"]
wdr_bus = ref_bus_name

inj_idx = int(buses.loc[buses["name"] == inj_bus, "bus_idx"].iloc[0])
wdr_idx = int(buses.loc[buses["name"] == wdr_bus, "bus_idx"].iloc[0])

p_test = np.zeros(len(buses))
p_test[inj_idx] = 1.0
p_test[wdr_idx] = -1.0
print("Sum of test injections (should be 0):", p_test.sum())

f_test = PTDF @ p_test

flow_test_df = pd.DataFrame({
    "line": lines["name"],
    "flow_for_test_transfer": f_test
})

print(f"\nTest transfer: +1 MW at {inj_bus}, -1 MW at {wdr_bus}")
display(flow_test_df.head(10))

Sum of test injections (should be 0): 0.0

Test transfer: +1 MW at NL0 18, -1 MW at NL0 0


,line,flow_for_test_transfer
0,0,-0.364179
1,1,-0.005967
2,10,-0.054766
3,11,0.054766
4,12,0.005967
5,13,-0.005967
6,14,-0.364179
7,15,-0.005579
8,16,-0.005967
9,17,0.418945


In [33]:
inj_idx = 18
slack_idx = ref_bus_idx

p = np.zeros(len(buses))
p[inj_idx] = 1.0
p[slack_idx] = -1.0

# remove slack entry from injections
p_red = np.delete(p, slack_idx)

# solve reduced DC load flow for voltage angles
theta_red = np.linalg.solve(B_red, p_red)

# reconstruct full angle vector
theta = np.insert(theta_red, slack_idx, 0.0)

# compute line flows from angles
f_dc = np.diag(b) @ A @ theta

# compute line flows from PTDF
f_ptdf = PTDF @ p

# compare
comparison_df = pd.DataFrame({
    "line": lines["name"].values,
    "flow_ptdf": f_ptdf,
    "flow_dc": f_dc,
    "difference": f_ptdf - f_dc
})

print("Max absolute difference:", np.max(np.abs(f_ptdf - f_dc)))
display(comparison_df)

Max absolute difference: 4.163336342344337e-17


,line,flow_ptdf,flow_dc,difference
0,0,-0.015743,-0.015743,6.938894e-18
1,1,-0.105750,-0.105750,1.387779e-17
2,10,0.029433,0.029433,1.040834e-17
3,11,-0.029433,-0.029433,1.734723e-17
4,12,0.105750,0.105750,1.387779e-17
5,13,-0.105750,-0.105750,-1.387779e-17
6,14,-0.015743,-0.015743,6.938894e-18
7,15,0.002999,0.002999,-7.372575e-18
8,16,-0.105750,-0.105750,1.387779e-17
9,17,-0.013691,-0.013691,1.734723e-18
